In [ ]:
import os
import sys
import re
from tqdm import tqdm
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

os.system('cls' if os.name == 'nt' else 'clear')

# 1. Setup
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_table_structure = True
pipeline_options.generate_page_images = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

source = "../documents/pdf/AF_III_02_01__Strojirenska_Firma.pdf"

if not os.path.exists(source):
    print(f"❌ Error: File '{source}' not found.")
    sys.exit(1)

print(f"🚀 Starting conversion of: {source}")

with tqdm(total=1, desc="Converting", bar_format="{desc}: |{bar}| {percentage:3.0f}% [{elapsed}]") as pbar:
    try:
        result = converter.convert(source)
        pbar.update(1)
    except Exception as e:
        print(f"\n❌ ERROR during conversion: {e}")
        sys.exit(1)


# --- 2. EXPORT WITH PHYSICAL READING ORDER ---
# Sort every element by its bounding-box position: (page_no, y) = top-of-page first.
# PDF coords: y=0 at bottom, y increases upward → we negate y so the sort is top-down.
#
# Figures are sorted by their BOTTOM edge (-bbox.b) rather than the top edge (-bbox.t).
# Reason: docling often reports a figure's bbox.t at the very top of the area it occupies,
# which can be above a sub-heading that visually follows it. Using bbox.b (the bottom of
# the figure) places the figure at the position where it ends, so text that comes after
# the figure in reading order sorts correctly after it.
# idx is kept as a tiebreaker to preserve docling's original iteration order when two
# elements sit at identical positions.

def export_with_reading_order(document):
    SKIP_LABELS = {'page_header', 'page_footer', 'page_number', 'footnote'}

    items = []
    for idx, (item, _level) in enumerate(document.iterate_items()):
        if not (hasattr(item, 'prov') and item.prov):
            continue
        prov = item.prov[0]
        label_val = item.label.value if hasattr(item.label, 'value') else str(item.label)
        # Figures: sort by bottom edge so they land after text that precedes them visually.
        # All other elements: sort by top edge (standard top-down reading order).
        sort_y = -prov.bbox.b if label_val == 'figure' else -prov.bbox.t
        items.append((prov.page_no, sort_y, idx, item))

    items.sort(key=lambda x: (x[0], x[1], x[2]))

    lines = []

    for _page, _y, _idx, item in items:
        label_val = item.label.value if hasattr(item.label, 'value') else str(item.label)

        if label_val in SKIP_LABELS:
            continue

        text = getattr(item, 'text', '').strip()

        if label_val == 'section_header':
            raw_level = getattr(item, 'level', 2)
            try:
                h = max(1, min(int(raw_level), 6))
            except (TypeError, ValueError):
                h = 2
            if lines and lines[-1].strip() != '':
                lines.append('')
            lines.append(f"{'#' * h} {text}")
            lines.append('')

        elif label_val == 'title':
            if lines and lines[-1].strip() != '':
                lines.append('')
            lines.append(f"# {text}")
            lines.append('')

        elif label_val == 'text':
            if text:
                lines.append(text)
                lines.append('')

        elif label_val == 'list_item':
            if text:
                lines.append(f"- {text}")

        elif label_val == 'figure':
            lines.append('<!-- image -->')
            lines.append('')

        elif label_val == 'table':
            try:
                lines.append(item.export_to_markdown(doc=document))
            except Exception:
                lines.append('<!-- table -->')
            lines.append('')

        elif label_val == 'caption':
            if text:
                lines.append(f"*{text}*")
                lines.append('')

        else:
            # Catch-all: preserve any other labelled text
            if text:
                lines.append(text)
                lines.append('')

    return '\n'.join(lines)


# --- 3. FIX HEADER LEVELS ---
# Ensures numbered headings get the right # depth (8.8.6.1 → ####),
# converts non-numbered headings to bold, and removes page-number lines.

def fix_headers(text):
    lines = text.split('\n')
    fixed = []

    for line in lines:
        # Remove page number lines  (- 123 -)
        if re.match(r'^\s*-\s+\d+\s+-\s*$', line):
            continue

        # Fix sub-bullets  (- o ...)
        m_sub = re.match(r'^\s*-\s*[oO]\s+(.*)', line)
        if m_sub:
            fixed.append(f"    - {m_sub.group(1)}")
            continue

        m_hdr = re.match(r'^\s*(#+)\s+(.*)', line)
        if m_hdr:
            content = m_hdr.group(2).strip()
            m_num = re.match(r'^((?:\d+\.)*\d+\.?)\s+(.*)', content)
            m_let = re.match(r'^([A-Z]\.)\s+(.*)', content)
            m_spc = re.match(r'^(Obsah|Závěr|Zdroje|Úvodní poznámky|Poznámky)', content, re.IGNORECASE)

            if fixed and fixed[-1].strip() != '':
                fixed.append('')

            if m_num:
                num = m_num.group(1)
                title = m_num.group(2)  # fixed: was group(3), regex only has 2 groups
                parts = [p for p in num.split('.') if p.strip()]
                level = max(1, min(len(parts), 6))
                fixed.append(f"{'#' * level} {num} {title}")
            elif m_let or m_spc:
                fixed.append(f"# {content}")
            else:
                fixed.append(f"**{content.strip('*')}**")

            fixed.append('')
        else:
            fixed.append(line)

    joined = '\n'.join(fixed)
    return re.sub(r'\n{3,}', '\n\n', joined)


# --- 4. REORDER CHAPTERS (safety net for any remaining out-of-order sections) ---
# Handles cases like "## 8.9" appearing between "#### 8.8.6.1" and "#### 8.8.6.2".

def reorder_sections(text):
    heading_re = re.compile(r'^(#{1,6}\s+)((?:\d+\.)*\d+\.?)\s+', re.MULTILINE)
    positions = [(m.start(), m.group(2)) for m in heading_re.finditer(text)]
    if not positions:
        return text

    preamble = text[:positions[0][0]]
    chunks = []
    for i, (start, num_str) in enumerate(positions):
        end = positions[i + 1][0] if i + 1 < len(positions) else len(text)
        key = tuple(int(x) for x in num_str.rstrip('.').split('.') if x.strip())
        chunks.append((key, text[start:end]))

    chunks.sort(key=lambda x: x[0])
    return preamble + ''.join(c for _, c in chunks)


# --- RUN PIPELINE ---
print("\n📐 Building markdown with physical reading order...")
md_output = export_with_reading_order(result.document)

print("🔧 Fixing header levels...")
md_output = fix_headers(md_output)

print("🔀 Reordering any remaining out-of-order chapters...")
md_output = reorder_sections(md_output)

output_filename = "../documents/md/AF_III_02_01__Strojirenska_Firma.md"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(md_output)

print(f"\n✅ Done! Saved as: {output_filename} ({len(md_output.splitlines())} lines)")

🚀 Starting conversion of: AF_III_02_01__Strojirenska_Firma.pdf


Converting: |██████████| 100% [02:40]



📐 Building markdown with physical reading order...
🔧 Fixing header levels...
🔀 Reordering any remaining out-of-order chapters...

✅ Done! Saved as: AF_III_02_01__Strojirenska_Firma2.md (12865 lines)
